In [1]:
# notebook to add promoter annotation and cgc annotation for updated predictions with indels
# cosmic ncv v98

In [2]:
# import packages
import pandas as pd
import os
import pybedtools
from collections import Counter
import numpy as np
from tqdm import tqdm

In [3]:
# add emVar status to predictions - no activity threshold, abs(skew) >= .5
# define function to add max ref/alt activities for a prediction DF for easier acitivity filtering
def add_max_act (pred_df):
    # k562
    pred_df['k562_max_act'] = [max([i,j], key=abs) for i,j in zip(pred_df['k562_ref_pred'],
                                                                  pred_df['k562_alt_pred'])]
    # hepg2
    pred_df['hepg2_max_act'] = [max([i,j], key=abs) for i,j in zip(pred_df['hepg2_ref_pred'],
                                                                   pred_df['hepg2_alt_pred'])]
    # sknsh
    pred_df['sknsh_max_act'] = [max([i,j], key=abs) for i,j in zip(pred_df['sknsh_ref_pred'],
                                                                   pred_df['sknsh_alt_pred'])]
    # add activity binary
    pred_df['k562_active'] = [1 if abs(i) >= 1 else 0 for i in pred_df['k562_max_act']]
    pred_df['hepg2_active'] = [1 if abs(i) >= 1 else 0 for i in pred_df['hepg2_max_act']]
    pred_df['sknsh_active'] = [1 if abs(i) >= 1 else 0 for i in pred_df['sknsh_max_act']]
    return pred_df
# define function to return emVar status - without activity
def add_emvar (pred_df,
               cell_type,
               skew_cutoff):
    # make a list for emvar status
    emvar_list = []
    for skew in pred_df[f'{cell_type}_skew_pred']:
        if abs(skew) > skew_cutoff:
            emvar_list.append(1)
        else:
            emvar_list.append(0)
    # add column to DF
    pred_df[f'{cell_type}_emvar'] = emvar_list
    return pred_df
# annotate emVars in predictions - without activity filter
def annotate_preds (df,
                    skew_thresh):
    # add max activity
    df = add_max_act(df)
    # add k562 emVar status
    df = add_emvar(df,
                   'k562',
                   skew_thresh)
    # add hepg2 emVar status
    df = add_emvar(df,
                   'hepg2',
                   skew_thresh)
    # add sknsh emVar status
    df = add_emvar(df,
                   'sknsh',
                   skew_thresh)
    # add emVar any column
    df['emvar_any'] = [1 if [i,j,k].count(1) > 0 else 0 for i,j,k in zip(df['k562_emvar'],
                                                                               df['hepg2_emvar'],
                                                                               df['sknsh_emvar'])]
    # add mean activity column
    df['mean_ref_act'] = [np.mean([k,h,s]) for k,h,s in zip(df['k562_ref_pred'],
                                                            df['hepg2_ref_pred'],
                                                            df['sknsh_ref_pred'])]
    # add mean skew column
    df['mean_skew'] = [np.mean([k,h,s]) for k,h,s in zip(df['k562_skew_pred'],
                                                         df['hepg2_skew_pred'],
                                                         df['sknsh_skew_pred'])]
    # add max skew column
    df['max_skew'] = [max([k,h,s], key=abs) for k,h,s in zip(df['k562_skew_pred'],
                                                         df['hepg2_skew_pred'],
                                                         df['sknsh_skew_pred'])]
    return df 

In [4]:
# open cosmic predictions
cosmic_preds_raw = pd.read_csv('../processed_data/mpac_preds/all.cosmic.v98.with.10bp.indels.017.vcf', sep = '\t')
# open cosmic TSV file
cosmic_tsv = pd.read_csv('../raw_data/CosmicNCV.tsv', sep = '\t')

In [5]:
# add emVar etc status to predictions
cosmic_preds = annotate_preds(cosmic_preds_raw, 0.5)

In [6]:
# add 'chr:pos:ref:alt' ID to predictions
cosmic_preds.loc[:,'var_id'] = [(':').join([i, str(j), k, l]) for i,j,k,l in zip(cosmic_preds['chrom'], 
                                                                                    cosmic_preds['pos'], 
                                                                                    cosmic_preds['ref'],
                                                                                    cosmic_preds['alt'])]

In [7]:
# function to take predictions and raw TSV and return DF with recurrence and predictions
# Recurrence is calculated from the COSMIC TSV by subsetting first on WGS Data and then counting
# presence of variant IDs, sample names are also included
def wgs_recurrence (tsv_df):
    #Subset TSV for wgs samples
    wgs_tsv = tsv_df[(tsv_df['Whole_Genome_Reseq'] == 'y') &
                              (tsv_df['Whole_Exome'] == 'n')]
    ids_recurrence = dict(Counter(wgs_tsv['GENOMIC_MUTATION_ID'].tolist()))
    return ids_recurrence

ids_recurrence = wgs_recurrence(cosmic_tsv)

In [8]:
# add recurrence to predictions
cosmic_preds.loc[:,'recurrence'] = [ids_recurrence.get(i) for i in cosmic_preds['id']]
# add a recurrent binary annotation
cosmic_preds.loc[:,'recurrent'] = [1 if i > 1 else 0 for i in cosmic_preds['recurrence']]

In [9]:
cosmic_preds.head()

,chrom,pos,id,ref,alt,k562_ref_pred,k562_alt_pred,k562_skew_pred,hepg2_ref_pred,hepg2_alt_pred,...,k562_emvar,hepg2_emvar,sknsh_emvar,emvar_any,mean_ref_act,mean_skew,max_skew,var_id,recurrence,recurrent
0,chr5,12057,COSV104949790,C,G,0.290299,0.484281,0.193983,0.236332,0.486731,...,0,0,0,0,0.219490,0.196977,0.250399,chr5:12057:C:G,1,0
1,chr5,12314,COSV106176984,C,T,3.124992,2.110848,-1.014144,2.444942,1.962961,...,1,0,0,1,2.854616,-0.619025,-1.014144,chr5:12314:C:T,1,0
2,chr5,13609,COSV64422606,C,T,0.165250,0.171957,0.006708,0.221217,0.226838,...,0,0,0,0,0.245531,-0.007524,-0.034900,chr5:13609:C:T,1,0
3,chr5,14173,COSV61294243,A,C,-0.059066,-0.023485,0.035582,-0.192251,-0.133702,...,0,0,0,0,-0.190478,0.025444,0.058550,chr5:14173:A:C,1,0
4,chr5,14196,COSV106241292,G,C,-0.066014,-0.059922,0.006092,-0.220630,-0.198749,...,0,0,0,0,-0.203165,0.020716,0.034173,chr5:14196:G:C,1,0


In [10]:
# make a bed file of cosmic variants for intersecting with promoter and other BED files
chrom4bed = []
start4bed = []
end4bed = []
id4bed = []
for chrom, pos, id, ref, alt in zip(cosmic_preds['chrom'], 
                                    cosmic_preds['pos'], 
                                    cosmic_preds['id'], 
                                    cosmic_preds['ref'], 
                                    cosmic_preds['alt']):
    # update chromosome and id lists
    chrom4bed.append(chrom)
    id4bed.append(id)
    # check the length of the variant for generating the start and stop intervals
    if len(ref) == 1 and len(alt) == 1: # SNPs
        start4bed.append(pos - 1)
        end4bed.append(pos)
    elif len(ref) < len(alt): # Insertions
        start4bed.append(pos)
        end4bed.append(pos)
    elif len(ref) > len(alt): # Deletions
        start4bed.append(pos)
        end4bed.append(pos + len(ref) - 1)
    
cosmic_bed = pybedtools.BedTool.from_dataframe(pd.DataFrame({0 : chrom4bed,
                                                             1 : start4bed,
                                                             2 : end4bed,
                                                             3 : id4bed}).drop_duplicates()).sort()

In [11]:
# open exon BED file
exonBed = pybedtools.BedTool('../raw_data/archive2/gencode.v44.basic.annotation.exons.splice.autosomes.v2.bed')
# get intersection with exons
cosmic_bed_exon_intersect = cosmic_bed.intersect(exonBed, u=True).to_dataframe()

In [12]:
len(cosmic_bed_exon_intersect)

586697

In [13]:
# open 250bp promoter bed file and intersect with exons for filtering
gencode_250bp = pybedtools.BedTool('../raw_data/archive2/gencode.v44.protein.coding.250bp.promoters.autosomes.v2.bed')
print(f'there are {len(gencode_250bp)} promoters in the bed file')
# intersect with exons
gencode_250bp_exon_filtered = gencode_250bp.subtract(exonBed)
gencode_250bp_exon_filtered_df = gencode_250bp_exon_filtered.to_dataframe()
print(f'there are {len(gencode_250bp_exon_filtered_df['name'].unique())} promoters remaining after filtering for exons')

there are 18340 promoters in the bed file
there are 17057 promoters remaining after filtering for exons


In [14]:
gencode_250bp_exon_filtered_df.head()

,chrom,start,end,name,score,strand,thickStart
0,chr1,65168,65418,OR4F5,0,+,ENSG00000186092.7_ENST00000641515.2_OR4F5
1,chr1,451678,451928,OR4F29,0,-,ENSG00000284733.2_ENST00000426406.4_OR4F29
2,chr1,686654,686904,OR4F16,0,-,ENSG00000284662.2_ENST00000332831.5_OR4F16
3,chr1,923672,923922,SAMD11,0,+,ENSG00000187634.13_ENST00000616016.5_SAMD11
4,chr1,959256,959506,NOC2L,0,-,ENSG00000188976.11_ENST00000327044.7_NOC2L


In [15]:
# open mueleman dhs data
mueleman_dhs_tsv = pd.read_csv('../raw_data/ENCFF503GCK.tsv', sep = '\t', low_memory=False)
# convert df to bed file
mueleman_dhs_bed = pybedtools.BedTool.from_dataframe(
    pd.DataFrame({
        0 : mueleman_dhs_tsv['seqname'],
        1 : mueleman_dhs_tsv['start'],
        2 : mueleman_dhs_tsv['end'],
        3 : mueleman_dhs_tsv['component']
    })
)

In [16]:
# open cornell non-coding cancer database
cnc_promoters = pd.read_csv('../raw_data/cnc.noncoding.driver.database.all.promoters.csv')
# make a bed file from the IDs in the cnc promoter file by filtering the full gencode promoter bed file
# get list of promoters
promoter_ids = list(cnc_promoters['Gene Name'].unique())
# filter the full promtoer bed file for only those ids
cnc_250_bed = pybedtools.BedTool.from_dataframe(gencode_250bp_exon_filtered_df[gencode_250bp_exon_filtered_df['name'].isin(promoter_ids)])

In [17]:
# get the intersection of cosmic variants and promoters
cosmic_250bp = cosmic_bed.intersect(gencode_250bp_exon_filtered, wa=True).to_dataframe()
# get the intersection of cosmic variants and dhs elements
cosmic_dhs = cosmic_bed.intersect(mueleman_dhs_bed, wa=True).to_dataframe()
# get the intersction of cosmic variants and cnc promoters
cosmic_cnc = cosmic_bed.intersect(cnc_250_bed, wa=True).to_dataframe()

In [18]:
# make dictionaries of each intersection for annotating predictions
# 250bp promoter
twoFifty_dict = dict(zip(cosmic_250bp['name'], [1 for i in range(len(cosmic_250bp))]))
# dhs
dhs_dict = dict(zip(cosmic_dhs['name'], [1 for i in range(len(cosmic_dhs))]))
# cnc promoter
cnc_dict = dict(zip(cosmic_cnc['name'], [1 for i in range(len(cosmic_cnc))]))
# exon intersect
exon_dict = dict(zip(cosmic_bed_exon_intersect['name'], [1 for i in range(len(cosmic_bed_exon_intersect))]))

In [19]:
sum(exon_dict.values())

586697

In [20]:
# open dELS seqlet bed files
# k562
k_dELS_seqlets = pybedtools.BedTool('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/analyses/repTF_k562_dELS_seqlets_01_vierstra_clusteded.bed')
# hepg2
h_dELS_seqlets = pybedtools.BedTool('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/analyses/repTF_hepg2_dELS_seqlets_01_vierstra_clusteded.bed')
# sknsh
s_dELS_seqlets = pybedtools.BedTool('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/analyses/repTF_sknsh_dELS_seqlets_01_vierstra_clusteded.bed')
# open dELS bed file
dELS_bed = pybedtools.BedTool('../../distal_cre_sat_mut/encode_cCRE_all/processed_data/GRCh38-dELS-only.bed')

In [21]:
# open promoter seqlet bed files
# k562
k_pro_seqlets = pybedtools.BedTool('../../promoter_sat_mut_comp/processed_data/bed_files/mpac_k562_merged_collapsed_repTFs_112225.bed')
# hepg2
h_pro_seqlets = pybedtools.BedTool('../../promoter_sat_mut_comp/processed_data/bed_files/mpac_hepg2_merged_collapsed_repTFs_112225.bed')
# sknsh
s_pro_seqlets = pybedtools.BedTool('../../promoter_sat_mut_comp/processed_data/bed_files/mpac_sknsh_merged_collapsed_repTFs_112225.bed')

In [22]:
# get the intersection of cosmic variants and promoter seqlets
# k562
cosmic_k_pro_seqlet = cosmic_bed.intersect(k_pro_seqlets, wa=True).to_dataframe()
# hepg2
cosmic_h_pro_seqlet = cosmic_bed.intersect(h_pro_seqlets, wa=True).to_dataframe()
# sknsh
cosmic_s_pro_seqlet = cosmic_bed.intersect(s_pro_seqlets, wa=True).to_dataframe()

In [23]:
# make dictionaries of each intersection for annotating predictions
# k562_seqlets
kProSeqlet_dict = dict(zip(cosmic_k_pro_seqlet['name'], [1 for i in range(len(cosmic_k_pro_seqlet))]))
# hepg2_seqlets
hProSeqlet_dict = dict(zip(cosmic_h_pro_seqlet['name'], [1 for i in range(len(cosmic_h_pro_seqlet))]))
# sknsh_seqlets
sProSeqlet_dict = dict(zip(cosmic_s_pro_seqlet['name'], [1 for i in range(len(cosmic_s_pro_seqlet))]))

In [24]:
# get the intersection of cosmic variants and dELS seqlets
# k562
cosmic_k_seqlet = cosmic_bed.intersect(k_dELS_seqlets, wa=True, wb=True).to_dataframe()
# hepg2
cosmic_h_seqlet = cosmic_bed.intersect(h_dELS_seqlets, wa=True, wb=True).to_dataframe()
# sknsh
cosmic_s_seqlet = cosmic_bed.intersect(s_dELS_seqlets, wa=True, wb=True).to_dataframe()
# intersect with distal elements
cosmic_dELS = cosmic_bed.intersect(dELS_bed, wa=True, wb=True).to_dataframe()

In [25]:
# make dictionaries of each intersection for annotating predictions
# k562_seqlets
kSeqlet_dict = dict(zip(cosmic_k_seqlet['name'], [1 for i in range(len(cosmic_k_seqlet))]))
# hepg2_seqlets
hSeqlet_dict = dict(zip(cosmic_h_seqlet['name'], [1 for i in range(len(cosmic_h_seqlet))]))
# sknsh_seqlets
sSeqlet_dict = dict(zip(cosmic_s_seqlet['name'], [1 for i in range(len(cosmic_s_seqlet))]))
# dELS
dELS_dict = dict(zip(cosmic_dELS['name'], [1 for i in range(len(cosmic_dELS))]))

In [26]:
# add annotations to df
# 250bp promoter
cosmic_preds.loc[:, '250bp_pro'] = [twoFifty_dict.get(i) if i in twoFifty_dict.keys() else 0 for i in cosmic_preds['id']]
# cnc promoter
cosmic_preds.loc[:, 'cnc_pro'] = [cnc_dict.get(i) if i in cnc_dict.keys() else 0 for i in cosmic_preds['id']]
# dhs
cosmic_preds.loc[:, 'dhs'] = [dhs_dict.get(i) if i in dhs_dict.keys() else 0 for i in cosmic_preds['id']]
# recurrent 250bp promoter
cosmic_preds.loc[:, 'recurrent_250bp_pro'] = [1 if sum([pro, recurrent]) == 2 else 0 for pro, recurrent in zip(cosmic_preds['250bp_pro'], cosmic_preds['recurrent'])]
# recurrent cnc promoter
cosmic_preds.loc[:, 'recurrent_cnc_pro'] = [1 if sum([pro, recurrent]) == 2 else 0 for pro, recurrent in zip(cosmic_preds['cnc_pro'], cosmic_preds['recurrent'])]
# indel
cosmic_preds.loc[:, 'indel'] = [1 if len(ref) > 1 or len(alt) > 1 else 0 for ref, alt in zip(cosmic_preds['ref'], cosmic_preds['alt'])]
# k562_seqlet
cosmic_preds.loc[:, 'k562_dELS_seqlet'] = [kSeqlet_dict.get(i) if i in kSeqlet_dict.keys() else 0 for i in cosmic_preds['id']]
# hepg2_seqlet
cosmic_preds.loc[:, 'hepg2_dELS_seqlet'] = [hSeqlet_dict.get(i) if i in hSeqlet_dict.keys() else 0 for i in cosmic_preds['id']]
# sknsh_seqlet
cosmic_preds.loc[:, 'sknsh_dELS_seqlet'] = [sSeqlet_dict.get(i) if i in sSeqlet_dict.keys() else 0 for i in cosmic_preds['id']]
# any seqlet
cosmic_preds.loc[:, 'dELS_seqlet'] = [1 if sum([k,h,s]) > 0 else 0 for k,h,s in zip(cosmic_preds['k562_dELS_seqlet'],
                                                                                    cosmic_preds['hepg2_dELS_seqlet'],
                                                                                    cosmic_preds['sknsh_dELS_seqlet'])]
# add dELS annotation
cosmic_preds.loc[:, 'encode_dELS'] = [dELS_dict.get(i) if i in dELS_dict.keys() else 0 for i in cosmic_preds['id']]
# add recurrent dELS
cosmic_preds.loc[:, 'recurrent_dELS'] = [1 if sum([dELS, recurrent]) == 2 else 0 for dELS, recurrent in zip(cosmic_preds['encode_dELS'], cosmic_preds['recurrent'])]
# k562 promoter seqlet
cosmic_preds.loc[:, 'k562_pro_seqlet'] = [kProSeqlet_dict.get(i) if i in kProSeqlet_dict.keys() else 0 for i in cosmic_preds['id']]
# hepg2 promoter seqlet
cosmic_preds.loc[:, 'hepg2_pro_seqlet'] = [hProSeqlet_dict.get(i) if i in hProSeqlet_dict.keys() else 0 for i in cosmic_preds['id']]
# sknsh promoter seqlet
cosmic_preds.loc[:, 'sknsh_pro_seqlet'] = [sProSeqlet_dict.get(i) if i in sProSeqlet_dict.keys() else 0 for i in cosmic_preds['id']]
# any promoter seqlet
cosmic_preds.loc[:, '250_pro_seqlet'] = [1 if sum([k,h,s]) > 0 else 0 for k,h,s in zip(cosmic_preds['k562_pro_seqlet'],
                                                                                       cosmic_preds['hepg2_pro_seqlet'],
                                                                                       cosmic_preds['sknsh_pro_seqlet'])]


In [27]:
# intersect vierstra collapsed seqlet calls with cosmic variants
# k562
k_cosmic_dels_v_seqlets = cosmic_bed.intersect(k_dELS_seqlets, wa=True, wb=True).to_dataframe()
# make a dictionary for annotating
# TF Family
kSeqletVierstraMatch = dict(zip(k_cosmic_dels_v_seqlets['name'], k_cosmic_dels_v_seqlets['itemRgb']))
# Enhancer ID
kEnhancerMatch = dict(zip(k_cosmic_dels_v_seqlets['name'], k_cosmic_dels_v_seqlets['blockSizes']))
# hepg2
h_cosmic_dels_v_seqlets = cosmic_bed.intersect(h_dELS_seqlets, wa=True, wb=True).to_dataframe()
# TF Family
hSeqletVierstraMatch = dict(zip(h_cosmic_dels_v_seqlets['name'], h_cosmic_dels_v_seqlets['itemRgb']))
# Enhancer ID
hEnhancerMatch = dict(zip(h_cosmic_dels_v_seqlets['name'], h_cosmic_dels_v_seqlets['blockSizes']))
# sknsh
s_cosmic_dels_v_seqlets = cosmic_bed.intersect(s_dELS_seqlets, wa=True, wb=True).to_dataframe()
# TF Family
sSeqletVierstraMatch = dict(zip(s_cosmic_dels_v_seqlets['name'], s_cosmic_dels_v_seqlets['itemRgb']))
# Enhancer ID
sEnhancerMatch = dict(zip(s_cosmic_dels_v_seqlets['name'], s_cosmic_dels_v_seqlets['blockSizes']))

In [28]:
# function to collapse cosmic TSV on primary histology by variant
def collapse_histology(tsv_df):
    """
    Collapse CosmicNCV data by GENOMIC_MUTATION_ID and determine primary histology.
    
    Primary histology is assigned based on max sample count for each variant.
    Ties are resolved by joining tied histologies with underscores (sorted alphabetically).

    Returns DataFrame with: GENOMIC_MUTATION_ID, primary_histology, all_histology
    """
    # count samples per variant per histology - pivot to wide format
    counts = tsv_df.groupby(['GENOMIC_MUTATION_ID', 'Primary histology']).size().unstack(fill_value=0)
    
    # get column names (histology types) sorted
    hist_cols = sorted(counts.columns.tolist())
    counts = counts[hist_cols]  # reorder columns alphabetically
    
    # get primary histology using idxmax
    primary_hist = counts.idxmax(axis=1)
    
    # identify and fix ties
    max_vals = counts.max(axis=1)
    tie_counts = (counts.eq(max_vals, axis=0) & counts.gt(0)).sum(axis=1)
    tie_mask = tie_counts > 1
    
    if tie_mask.any():
        tie_idx = counts.index[tie_mask]
        tie_data = counts.loc[tie_idx]
        tie_maxes = max_vals.loc[tie_idx]
        
        # vectorized tie handling
        is_max = tie_data.eq(tie_maxes, axis=0) & tie_data.gt(0)
        tied_histologies = is_max.apply(lambda row: ':'.join(counts.columns[row].tolist()), axis=1)
        primary_hist.loc[tie_idx] = tied_histologies
    
    # all_histology: use numpy for speed
    has_hist = (counts > 0).values
    hist_array = np.array(hist_cols)
    all_hist_list = [','.join(hist_array[row]) for row in has_hist]
    
    # build result
    result = pd.DataFrame({
        'cosmic_id': counts.index,
        'primary_histology': primary_hist.values,
        'all_histology': all_hist_list
    })
    
    return result

In [29]:
# assign primary histology to each variant
cosmic_var_histology = collapse_histology(cosmic_tsv)

In [30]:
# make dictionaries of histology id pairs
cosmicPrimaryHistDict = dict(zip(cosmic_var_histology['cosmic_id'], cosmic_var_histology['primary_histology']))
cosmicAllHistDict = dict(zip(cosmic_var_histology['cosmic_id'], cosmic_var_histology['all_histology']))

In [31]:
# add tf family id to preds
# k562
cosmic_preds.loc[:, 'k562_dELS_seqlet_tf_fam'] = [kSeqletVierstraMatch.get(i) if i in kSeqletVierstraMatch.keys() else 0 for i in cosmic_preds['id']]
# hepg2
cosmic_preds.loc[:, 'hepg2_dELS_seqlet_tf_fam'] = [hSeqletVierstraMatch.get(i) if i in hSeqletVierstraMatch.keys() else 0 for i in cosmic_preds['id']]
# sknsh
cosmic_preds.loc[:, 'sknsh_dELS_seqlet_tf_fam'] = [sSeqletVierstraMatch.get(i) if i in sSeqletVierstraMatch.keys() else 0 for i in cosmic_preds['id']]

In [32]:
# add enhancer ID intersection
# k562
cosmic_preds.loc[:, 'k562_dELS_seqlet_enh_ID'] = [kEnhancerMatch.get(i) if i in kEnhancerMatch.keys() else 0 for i in cosmic_preds['id']]
# hepg2
cosmic_preds.loc[:, 'hepg2_dELS_seqlet_enh_ID'] = [hEnhancerMatch.get(i) if i in hEnhancerMatch.keys() else 0 for i in cosmic_preds['id']]
# sknsh
cosmic_preds.loc[:, 'sknsh_dELS_seqlet_enh_ID'] = [sEnhancerMatch.get(i) if i in sEnhancerMatch.keys() else 0 for i in cosmic_preds['id']]

In [33]:
# add histology information
cosmic_preds.loc[:, 'primary_histology'] = [cosmicPrimaryHistDict.get(i) for i in cosmic_preds['id']]
cosmic_preds.loc[:, 'all_histology'] = [cosmicAllHistDict.get(i) for i in cosmic_preds['id']]

In [34]:
# open annotations
vepExonAnnotations = pd.read_csv('../processed_data/ensembl_vep_exon_overlap_most_severe.txt',
                                 sep = '\t')
# build a dictionary for adding the vep most severe 
vepDict = dict(zip(
    vepExonAnnotations['cosmic_id'],
    vepExonAnnotations['vep_most_severe']
))
# add to df
cosmic_preds.loc[:, 'vep_most_severe'] = [vepDict.get(i) if i in vepDict.keys() else 0 for i in cosmic_preds['id']]
# add coding related binary column
exonic_coding = [
        "transcript_ablation", "stop_gained", "frameshift_variant",
        "stop_lost", "start_lost", "inframe_insertion", "inframe_deletion",
        "missense_variant", "protein_altering_variant",
        "incomplete_terminal_codon_variant", "start_retained_variant",
        "stop_retained_variant", "synonymous_variant", "coding_sequence_variant",
    ]
cosmic_preds.loc[:, 'vep_coding_related'] = [1 if i in exonic_coding else 0 for i in cosmic_preds['vep_most_severe']]

In [35]:
# filter for only non-coding and synonymous variants before saving
cosmic_preds_nonSyn = cosmic_preds[(cosmic_preds['vep_coding_related'] == 0) |
                                   (cosmic_preds['vep_most_severe'] == 'synonymous_variant')]

In [36]:
len(cosmic_preds_nonSyn)

2190594

In [37]:
cosmic_preds_nonSyn['250bp_pro'].sum()

np.int64(5908)

In [38]:
cosmic_preds_nonSyn['cnc_pro'].sum()

np.int64(327)

In [39]:
cosmic_preds.iloc[0,:]

chrom                                 chr5
pos                                  12057
id                           COSV104949790
ref                                      C
alt                                      G
k562_ref_pred                     0.290299
k562_alt_pred                     0.484281
k562_skew_pred                    0.193983
hepg2_ref_pred                    0.236332
hepg2_alt_pred                    0.486731
hepg2_skew_pred                   0.250399
sknsh_ref_pred                    0.131839
sknsh_alt_pred                    0.278388
sknsh_skew_pred                    0.14655
k562_max_act                      0.484281
hepg2_max_act                     0.486731
sknsh_max_act                     0.278388
k562_active                              0
hepg2_active                             0
sknsh_active                             0
k562_emvar                               0
hepg2_emvar                              0
sknsh_emvar                              0
emvar_any  

In [40]:
# # save annotated predictions to df for analysis
# cosmic_preds.to_csv('../processed_data/all.cosmic.v98.autosome.with.10bp.indels.annotated.017.031826.tsv',
#                     sep = '\t',
#                     index = False)
# # save bed file to disk for distal element sat mut example hunting
# cosmic_bed.to_dataframe().to_csv('../processed_data/all.cosmic.v98.autosome.with.10bp.indels.bed', sep = '\t', header = None, index = False)